In [0]:
%run ../../_shared/00_project_config

In [0]:
from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Source-specific values for this Bronze dataset.
# Project-wide paths and schema names are loaded from 00_project_config.

SOURCE_SYSTEM = "API-Football"

FILE_PATH = (
    f"{API_FOOTBALL_RAW_ROOT}"
    "/venue_data/raw_venues.json"
)

BRONZE_TABLE = f"{BRONZE_SCHEMA}.api_venues_data"


# -----------------------------------------------------------------------------
# Read raw API-Football venue data
# -----------------------------------------------------------------------------
# The source file contains multi-line JSON with venue-level attributes such as:
#   - venue ID
#   - venue name
#   - address
#   - city
#   - country
#   - capacity
#   - surface
#   - image URL
#
# Bronze preserves the source structure and source datatypes as closely as
# possible. No cleaning, casting, deduplication, or business validation is
# performed here; those responsibilities belong in the Silver layer.
#
# Technical metadata is added for lineage and traceability:
#   _ingested_at   - timestamp of this Bronze load
#   _source_file   - actual Unity Catalog Volume file that produced the row
#   _source_system - originating external system

bronze_df = (
    spark.read
    .option("multiLine", "true")
    .json(FILE_PATH)
    .withColumn(
        "_ingested_at",
        F.current_timestamp(),
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path"),
    )
    .withColumn(
        "_source_system",
        F.lit(SOURCE_SYSTEM),
    )
)


# -----------------------------------------------------------------------------
# Write to Bronze Delta table
# -----------------------------------------------------------------------------
# Append mode preserves ingestion history.
#
# mergeSchema is intentionally enabled in Bronze so additive source-schema
# changes can be captured without immediately breaking ingestion. Schema
# standardization and business-level validation are handled in Silver.

(
    bronze_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(f"Successfully appended data to Bronze table: {BRONZE_TABLE}")